# 1) Setup

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("..")/"data"/"cybersecurity_intrusion_data.csv"

# Load data
intrusion_df = pd.read_csv(data_path, keep_default_na=False)

In [2]:
intrusion_df

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0
...,...,...,...,...,...,...,...,...,...,...,...
9532,SID_09533,194,ICMP,3,226.049889,AES,0.517737,3,Chrome,0,1
9533,SID_09534,380,TCP,3,182.848475,None,0.408485,0,Chrome,0,0
9534,SID_09535,664,TCP,5,35.170248,AES,0.359200,1,Firefox,0,0
9535,SID_09536,406,TCP,4,86.664703,AES,0.537417,1,Chrome,1,0


In [3]:
intrusion_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           9537 non-null   object 
 1   network_packet_size  9537 non-null   int64  
 2   protocol_type        9537 non-null   object 
 3   login_attempts       9537 non-null   int64  
 4   session_duration     9537 non-null   float64
 5   encryption_used      9537 non-null   object 
 6   ip_reputation_score  9537 non-null   float64
 7   failed_logins        9537 non-null   int64  
 8   browser_type         9537 non-null   object 
 9   unusual_time_access  9537 non-null   int64  
 10  attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 819.7+ KB


In [4]:
intrusion_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
session_id,9537,9537,SID_09537,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
network_packet_size,9537.0,NaN,NaN,NaN,500.430639,198.379364,64.0,365.0,499.0,635.0,1285.0
protocol_type,9537,3,TCP,6624,NaN,NaN,NaN,NaN,NaN,NaN,NaN
login_attempts,9537.0,NaN,NaN,NaN,4.032086,1.963012,1.0,3.0,4.0,5.0,13.0
session_duration,9537.0,NaN,NaN,NaN,792.745312,786.560144,0.5,231.953006,556.277457,1105.380602,7190.392213
encryption_used,9537,3,AES,4706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_reputation_score,9537.0,NaN,NaN,NaN,0.331338,0.177175,0.002497,0.191946,0.314778,0.453388,0.924299
failed_logins,9537.0,NaN,NaN,NaN,1.517773,1.033988,0.0,1.0,1.0,2.0,5.0
browser_type,9537,5,Chrome,5137,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unusual_time_access,9537.0,NaN,NaN,NaN,0.149942,0.357034,0.0,0.0,0.0,0.0,1.0


In [5]:
print("Missing values per column:\n", intrusion_df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", intrusion_df.duplicated().sum())
print()
intrusion_df["attack_detected"].value_counts()

Missing values per column:
 session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64

Duplicate rows: 0



attack_detected
0    5273
1    4264
Name: count, dtype: int64

# 2) Cleaning data

In [6]:
# Drop session id column
intrusion_df = intrusion_df.drop(columns=["session_id"])
intrusion_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   network_packet_size  9537 non-null   int64  
 1   protocol_type        9537 non-null   object 
 2   login_attempts       9537 non-null   int64  
 3   session_duration     9537 non-null   float64
 4   encryption_used      9537 non-null   object 
 5   ip_reputation_score  9537 non-null   float64
 6   failed_logins        9537 non-null   int64  
 7   browser_type         9537 non-null   object 
 8   unusual_time_access  9537 non-null   int64  
 9   attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 745.2+ KB


# 3) Perceptron

In [7]:
# Seperate label from df
percep_y = intrusion_df["attack_detected"].astype(int)
percep_x = intrusion_df.drop(columns=["attack_detected"]).copy()

# Split column types
percep_category_x = percep_x.select_dtypes(include=["object", "category"]).columns
percep_num_x = percep_x.select_dtypes(exclude=["object", "category"]).columns

# Display list
print("Categorial Columns", list(percep_category_x))
print("Numeric Columns", list(percep_num_x))

Categorial Columns ['protocol_type', 'encryption_used', 'browser_type']
Numeric Columns ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access']


In [8]:
# Convert categorial columns into a 0-1 encoding
category_encoded_col_x = pd.get_dummies(percep_x, columns=percep_category_x, drop_first=False, dtype=int)
print("Before:", percep_x.shape, "After:", category_encoded_col_x.shape)
category_encoded_col_x.head()

Before: (9537, 9) After: (9537, 17)


,network_packet_size,login_attempts,session_duration,ip_reputation_score,failed_logins,unusual_time_access,protocol_type_ICMP,protocol_type_TCP,protocol_type_UDP,encryption_used_AES,encryption_used_DES,encryption_used_None,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown
0,599,4,492.983263,0.606818,1,0,0,1,0,0,1,0,0,1,0,0,0
1,472,3,1557.996461,0.301569,0,0,0,1,0,0,1,0,0,0,1,0,0
2,629,3,75.044262,0.739164,2,0,0,1,0,0,1,0,1,0,0,0,0
3,804,4,601.248835,0.123267,0,0,0,0,1,0,1,0,0,0,0,0,1
4,453,5,532.540888,0.054874,1,0,0,1,0,1,0,0,0,0,1,0,0


In [9]:
# Create Perceptron model
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import classification_report, confusion_matrix

precep_x_train, precep_x_test, precep_y_train, precep_y_test = train_test_split(category_encoded_col_x, 
                                                                                percep_y, test_size=0.2,
                                                                                random_state=29,
                                                                                stratify=percep_y)

perceptron_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Perceptron(random_state=29))
])

perceptron_clf.fit(precep_x_train, precep_y_train)
precep_y_prediction = perceptron_clf.predict(precep_x_test)

print("Confusion matrix:\n", confusion_matrix(precep_y_test, precep_y_prediction))
print("\nReport:\n", classification_report(precep_y_test, precep_y_prediction))

Confusion matrix:
 [[646 409]
 [326 527]]

Report:
               precision    recall  f1-score   support

           0       0.66      0.61      0.64      1055
           1       0.56      0.62      0.59       853

    accuracy                           0.61      1908
   macro avg       0.61      0.62      0.61      1908
weighted avg       0.62      0.61      0.62      1908

